# Couche Silver KBO Open Data : `entreprise` → `entreprise_silver`

## 0. Contexte : bronze vs silver

La couche **bronze** (`entreprise`) reste sous forme **brute** :

- des codes non traduits (`Status="AC"`, `TypeOfAddress="REGO"`...) ;
- des tableaux indexés `0/1/2/3` sans clé porteuse de sens ;
- des champs dupliqués par langue (`CountryNL`/`CountryFR`, `MunicipalityNL`/`MunicipalityFR`...) ;
- des activités qui réapparaissent en double sous plusieurs versions NACE (2003, 2008, 2025) pour la même réalité.

La couche **silver** de transformation géres tout ça

## 1. Lecture du bronze

Charger `entreprise` et `kbo_code` filtré sur `Language="FR"`.

In [1]:
import os
import re
from pathlib import Path
from pprint import pprint
from __future__ import annotations
import pymongo

MONGO_URI = os.environ.get("MONGO_URI", "mongodb://localhost:27017")
DB_NAME = os.environ.get("MONGO_DB", "kbo")

client = pymongo.MongoClient(MONGO_URI)
db = client[DB_NAME]

# Référentiel de traduction FR : {(Category, Code): Description}
code_map = {
    (doc["Category"], doc["Code"]): doc["Description"]
    for doc in db.code.find({"Language": "FR"})
}


def translate(category: str, raw_code: str):
    """Traduit `raw_code` dans `category` via le référentiel kbo_code (FR).

    Renvoie None si `raw_code` est vide (l'appelant doit alors omettre le champ
    plutôt que d'écrire une traduction nulle). Si le code n'est pas trouvé dans
    le référentiel, on retombe sur le code brut plutôt que de planter.
    """
    if not raw_code:
        return None
    return code_map.get((category, raw_code), raw_code)


print(f"code_map: {len(code_map):,} entrées (catégories FR)")
print(f"enterprise (bronze): {db.enterprise.count_documents({}):,} documents")


code_map: 10,641 entrées (catégories FR)
enterprise (bronze): 1,955,776 documents


## 2. Champs scalaires codés

`Status`, `JuridicalSituation`, `TypeOfEnterprise`, `JuridicalForm`, `JuridicalFormCAC` : cinq champs plats, chacun traduit indépendamment via `kbo_code`. Un champ absent ou vide dans le bronze (ex. `JuridicalFormCAC=""`) doit être **omis** du silver plutôt que d'y apparaître traduit en valeur nulle.

In [2]:
SCALAR_CODE_FIELDS = {
    "Status": "status",
    "JuridicalSituation": "juridicalSituation",
    "TypeOfEnterprise": "typeOfEnterprise",
    "JuridicalForm": "juridicalForm",
    # code.csv n'a pas de catégorie "JuridicalFormCAC" dédiée : on réutilise le
    # référentiel "JuridicalForm", les deux champs partageant la même liste de codes.
    "JuridicalFormCAC": "juridicalFormCAC",
}


def build_scalar_fields(doc: dict) -> dict:
    """Traduit les 5 champs scalaires codés du bronze ; un champ absent ou vide
    (ex. JuridicalFormCAC="") est omis plutôt que traduit en valeur nulle."""
    out = {}
    for bronze_field, silver_field in SCALAR_CODE_FIELDS.items():
        raw = doc.get(bronze_field, "")
        if not raw:
            continue
        category = "JuridicalForm" if bronze_field == "JuridicalFormCAC" else bronze_field
        out[silver_field] = translate(category, raw)
    return out


def build_silver_header(doc: dict) -> dict:
    """_id / enterpriseNumber / startDate + champs scalaires traduits : socle
    commun à toute entité de premier niveau (une entreprise)."""
    header = {
        "_id": doc["_id"],
        "enterpriseNumber": doc.get("EnterpriseNumber", doc["_id"]),
    }
    if doc.get("StartDate"):
        header["startDate"] = doc["StartDate"]
    header.update(build_scalar_fields(doc))
    return header


sample_enterprise = db.enterprise.find_one({"_id": "0200.245.711"})
pprint(build_silver_header(sample_enterprise))


{'_id': '0200.245.711',
 'enterpriseNumber': '0200.245.711',
 'juridicalForm': 'Société coopérative de droit public (ancien statut)',
 'juridicalSituation': 'Dissolution volontaire – liquidation',
 'startDate': '01-01-1922',
 'status': 'Actif',
 'typeOfEnterprise': 'Personne morale'}


In [3]:
# Repère : sortie attendue à ce stade, sur un exemple connu
# (comparez-la visuellement au résultat de la cellule précédente)
{
  "_id": "0200.245.711",
  "enterpriseNumber": "0200.245.711",
  "startDate": "01-01-1922",
  "juridicalForm": "Société coopérative de droit public (ancien statut)",
  "juridicalSituation": "Dissolution volontaire – liquidation",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

{'_id': '0200.245.711',
 'enterpriseNumber': '0200.245.711',
 'startDate': '01-01-1922',
 'juridicalForm': 'Société coopérative de droit public (ancien statut)',
 'juridicalSituation': 'Dissolution volontaire – liquidation',
 'status': 'Actif',
 'typeOfEnterprise': 'Personne morale'}

## 3. Dénominations : tableau → dict `{type traduit: {language, denomination}}`

Chaque entrée doit être keyée par son `TypeOfDenomination` **traduit**

In [4]:
def build_denominations(entity_number: str) -> dict:
    """{type traduit: {language, denomination}} ; dernier gagne en cas de type dupliqué
    (une entrée écrase simplement la précédente à la même clé)."""
    out = {}
    for row in db.denomination.find({"EntityNumber": entity_number}):
        type_label = translate("TypeOfDenomination", row["TypeOfDenomination"])
        out[type_label] = {
            "language": translate("Language", row["Language"]),
            "denomination": row["Denomination"],
        }
    return out


pprint(build_denominations("0200.245.711"))


{'Abréviation': {'denomination': 'DENDEROORD', 'language': 'néerlandais'},
 'Dénomination': {'denomination': 'Intercommunaal Sanatorium Denderoord',
                  'language': 'néerlandais'}}


In [5]:
# Repère : sortie attendue à ce stade, sur un exemple connu
# (comparez-la visuellement au résultat de la cellule précédente)
{
  "_id": "0200.245.711",
  "enterpriseNumber": "0200.245.711",
  "startDate": "01-01-1922",
  "denominations": {
    "Abréviation": {
      "language": "néerlandais",
      "denomination": "DENDEROORD"
    },
    "Dénomination": {
      "language": "néerlandais",
      "denomination": "Intercommunaal Sanatorium Denderoord"
    }
  },
  "juridicalForm": "Société coopérative de droit public (ancien statut)",
  "juridicalSituation": "Dissolution volontaire – liquidation",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

{'_id': '0200.245.711',
 'enterpriseNumber': '0200.245.711',
 'startDate': '01-01-1922',
 'denominations': {'Abréviation': {'language': 'néerlandais',
   'denomination': 'DENDEROORD'},
  'Dénomination': {'language': 'néerlandais',
   'denomination': 'Intercommunaal Sanatorium Denderoord'}},
 'juridicalForm': 'Société coopérative de droit public (ancien statut)',
 'juridicalSituation': 'Dissolution volontaire – liquidation',
 'status': 'Actif',
 'typeOfEnterprise': 'Personne morale'}

## 4. Adresses : tableau → dict `{type traduit: {country, street...}}`

Même principe clé-traduite que les dénominations, plus deux règles spécifiques :

- **`country`** : `CountryFR` nettoyé des mentions entre parenthèses (`"France (Métropole)"` → `"France"`) et des espaces multiples ; si le résultat est vide, mettre `"Belgique"` par défaut.
- **Champs vides omis** : `box`, `zipcode`... vides ne doivent pas apparaître dans le document silver.

In [6]:
def clean_country(raw_country: str) -> str:
    """Nettoie CountryFR : retire les mentions entre parenthèses et les espaces
    multiples ; renvoie "Belgique" par défaut si le résultat est vide (cas des
    adresses belges, où CountryFR n'est simplement jamais renseigné)."""
    cleaned = re.sub(r"\([^)]*\)", "", raw_country)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned or "Belgique"


def build_addresses(entity_number: str) -> dict:
    """{type traduit: {country, zipcode, municipality, street, houseNumber, box}} ;
    on privilégie systématiquement les colonnes *FR* (StreetFR, MunicipalityFR),
    et les champs vides (zipcode, municipality, street, houseNumber, box) sont omis."""
    out = {}
    for row in db.address.find({"EntityNumber": entity_number}):
        type_label = translate("TypeOfAddress", row["TypeOfAddress"])
        address = {"country": clean_country(row.get("CountryFR", ""))}
        for bronze_field, silver_field in [
            ("Zipcode", "zipcode"),
            ("MunicipalityFR", "municipality"),
            ("StreetFR", "street"),
            ("HouseNumber", "houseNumber"),
            ("Box", "box"),
        ]:
            value = row.get(bronze_field, "")
            if value:
                address[silver_field] = value
        out[type_label] = address
    return out


pprint(build_addresses("0200.245.711"))


{'Siège': {'country': 'Belgique',
           'houseNumber': '247',
           'municipality': 'Geraardsbergen',
           'street': 'Hoge Buizemont',
           'zipcode': '9500'}}


In [7]:
# Repère : sortie attendue à ce stade, sur un exemple connu
# (comparez-la visuellement au résultat de la cellule précédente)
{
  "_id": "0200.245.711",
  "enterpriseNumber": "0200.245.711",
  "startDate": "01-01-1922",
  "denominations": {
    "Abréviation": {
      "language": "néerlandais",
      "denomination": "DENDEROORD"
    },
    "Dénomination": {
      "language": "néerlandais",
      "denomination": "Intercommunaal Sanatorium Denderoord"
    }
  },
  "addresses": {
    "Siège": {
      "country": "Belgique",
      "zipcode": "9500",
      "municipality": "Geraardsbergen",
      "street": "Hoge Buizemont",
      "houseNumber": "247",
      "box": ""
    }
  },
  "juridicalForm": "Société coopérative de droit public (ancien statut)",
  "juridicalSituation": "Dissolution volontaire – liquidation",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

{'_id': '0200.245.711',
 'enterpriseNumber': '0200.245.711',
 'startDate': '01-01-1922',
 'denominations': {'Abréviation': {'language': 'néerlandais',
   'denomination': 'DENDEROORD'},
  'Dénomination': {'language': 'néerlandais',
   'denomination': 'Intercommunaal Sanatorium Denderoord'}},
 'addresses': {'Siège': {'country': 'Belgique',
   'zipcode': '9500',
   'municipality': 'Geraardsbergen',
   'street': 'Hoge Buizemont',
   'houseNumber': '247',
   'box': ''}},
 'juridicalForm': 'Société coopérative de droit public (ancien statut)',
 'juridicalSituation': 'Dissolution volontaire – liquidation',
 'status': 'Actif',
 'typeOfEnterprise': 'Personne morale'}

## 5. Contacts : tableau → dict `{email, phone, web}`

`EntityContact` (indique juste si le contact appartient à l'entreprise, un établissement ou une succursale) ne doit **jamais** être repris dans le silver. Un contact avec une valeur vide doit être ignoré.

In [8]:
# TEL -> "phone" (et non "tel") : seul renommage qui ne soit pas une simple
# mise en minuscule du ContactType brut.
CONTACT_TYPE_TO_FIELD = {"EMAIL": "email", "TEL": "phone", "WEB": "web", "FAX": "fax"}


def build_contacts(entity_number: str) -> dict:
    """{email?, phone?, web?, fax?} ; EntityContact jamais lu (il ne fait qu'indiquer
    le niveau -- entreprise/établissement/succursale -- pas une info de contact),
    et une valeur vide est ignorée."""
    out = {}
    for row in db.contact.find({"EntityNumber": entity_number}):
        value = row.get("Value", "")
        if not value:
            continue
        field = CONTACT_TYPE_TO_FIELD.get(row["ContactType"], row["ContactType"].lower())
        out[field] = value
    return out


pprint(build_contacts("0201.543.234"))


{'email': 'officiel.ic-tibi@tibi.be',
 'fax': '071 36 04 84',
 'phone': '071 44 00 40'}


In [9]:
# Repère : sortie attendue à ce stade, sur un exemple connu
# (comparez-la visuellement au résultat de la cellule précédente)
{
  "_id": "0201.543.234",
  "enterpriseNumber": "0201.543.234",
  "startDate": "26-10-1948",
  "denominations": {
    "Dénomination": {
      "language": "français",
      "denomination": "TIBI"
    }
  },
  "addresses": {
    "Siège": {
      "country": "Belgique",
      "zipcode": "6010",
      "municipality": "Charleroi",
      "street": "Rue du Déversoir",
      "houseNumber": "1",
      "box": ""
    }
  },
  "contacts": {
    "phone": "071 44 00 40",
    "email": "officiel.ic-tibi@tibi.be",
    "fax": "071 36 04 84"
  },
  "juridicalForm": "Société coopérative de droit public",
  "juridicalSituation": "Situation normale",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

{'_id': '0201.543.234',
 'enterpriseNumber': '0201.543.234',
 'startDate': '26-10-1948',
 'denominations': {'Dénomination': {'language': 'français',
   'denomination': 'TIBI'}},
 'addresses': {'Siège': {'country': 'Belgique',
   'zipcode': '6010',
   'municipality': 'Charleroi',
   'street': 'Rue du Déversoir',
   'houseNumber': '1',
   'box': ''}},
 'contacts': {'phone': '071 44 00 40',
  'email': 'officiel.ic-tibi@tibi.be',
  'fax': '071 36 04 84'},
 'juridicalForm': 'Société coopérative de droit public',
 'juridicalSituation': 'Situation normale',
 'status': 'Actif',
 'typeOfEnterprise': 'Personne morale'}

## 6. Activités : dédoublonnage inter-versions NACE + répartition main/secondary

Le point le plus subtil de toute la couche silver. Une même activité réelle est souvent codée sous **plusieurs versions NACE** (2003, 2008, 2025) avec des libellés différents mais qui décrivent la même chose. Règle : dédoublonner sur `(activityGroup, description)` et, en cas de collision, **garder la version NACE la plus récente**. Le `NaceCode` brut ne doit **jamais** être gardé dans le silver, une fois `description` résolue via `Nace{version}`, le code numérique ne sert plus à un lecteur humain. Répartir le résultat en `{main: [...], secondary: [...]}` selon `Classification`.

In [10]:
def build_activities(entity_number: str) -> dict:
    """Dédoublonne les activités sur (activityGroup traduit, description traduite) en
    gardant, en cas de collision, la version NACE la plus récente. Répartit ensuite
    le résultat en {main: [...], secondary: [...]} selon Classification.
    NaceCode brut n'est jamais conservé : une fois `description` résolue via
    Nace{version}, il ne sert plus à un lecteur humain."""
    best = {}  # (activityGroup, description) -> ligne la plus récente vue jusqu'ici
    for row in db.activity.find({"EntityNumber": entity_number}):
        nace_version = row["NaceVersion"]
        description = translate(f"Nace{nace_version}", row["NaceCode"])
        activity_group = translate("ActivityGroup", row["ActivityGroup"])
        key = (activity_group, description)
        if key not in best or int(nace_version) > int(best[key]["naceVersion"]):
            best[key] = {
                "activityGroup": activity_group,
                "description": description,
                "naceVersion": nace_version,
                "classification": row["Classification"],
            }

    result = {"main": [], "secondary": []}
    for activity in best.values():
        bucket = "main" if activity["classification"] == "MAIN" else "secondary"
        result[bucket].append({
            "activityGroup": activity["activityGroup"],
            "description": activity["description"],
            "naceVersion": activity["naceVersion"],
        })
    return result


pprint(build_activities("0201.311.226"))


{'main': [{'activityGroup': 'Activités ONSS',
           'description': 'Distribution d’électricité',
           'naceVersion': '2025'},
          {'activityGroup': 'Activités TVA',
           'description': 'Distribution d’électricité',
           'naceVersion': '2025'},
          {'activityGroup': 'Activités ONSS',
           'description': "Distribution d'électricité",
           'naceVersion': '2008'},
          {'activityGroup': 'Activités TVA',
           'description': "Distribution d'électricité",
           'naceVersion': '2008'}],
 'secondary': [{'activityGroup': 'Activités TVA',
                'description': 'Distribution de combustibles gazeux par '
                               'conduites',
                'naceVersion': '2025'},
               {'activityGroup': 'Activités TVA',
                'description': 'Production d’électricité à partir de sources '
                               'non renouvelables',
                'naceVersion': '2025'},
               {'activit

In [11]:
# Repère : sortie attendue à ce stade, sur un exemple connu
# (comparez-la visuellement au résultat de la cellule précédente)
{
  "_id": "0201.311.226",
  "enterpriseNumber": "0201.311.226",
  "startDate": "01-01-1968",
  "denominations": {
    "Dénomination": {
      "language": "néerlandais",
      "denomination": "FLUVIUS"
    }
  },
  "addresses": {
    "Siège": {
      "country": "Belgique",
      "zipcode": "3500",
      "municipality": "Hasselt",
      "street": "Trichterheideweg",
      "houseNumber": "8",
      "box": ""
    }
  },
  "contacts": {},
  "activities": {
    "main": [
      {
        "activityGroup": "Activités TVA",
        "description": "Distribution d'électricité",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Distribution d’électricité",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités ONSS",
        "description": "Distribution d'électricité",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités ONSS",
        "description": "Distribution d’électricité",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Production d'électricité",
        "naceVersion": "2003"
      }
    ],
    "secondary": [
      {
        "activityGroup": "Activités TVA",
        "description": "Activités de télécommunications filaires, sans fil et satellitaires",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Production d’électricité à partir de sources non renouvelables",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Collecte et traitement des eaux usées",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Distribution de combustibles gazeux par conduites",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Production d'électricité",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Télécommunications sans fil",
        "naceVersion": "2008"
      }
    ]
  },

  "branches": {},
  "juridicalForm": "Association chargée de mission (Région flamande)",
  "juridicalSituation": "Situation normale",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

{'_id': '0201.311.226',
 'enterpriseNumber': '0201.311.226',
 'startDate': '01-01-1968',
 'denominations': {'Dénomination': {'language': 'néerlandais',
   'denomination': 'FLUVIUS'}},
 'addresses': {'Siège': {'country': 'Belgique',
   'zipcode': '3500',
   'municipality': 'Hasselt',
   'street': 'Trichterheideweg',
   'houseNumber': '8',
   'box': ''}},
 'contacts': {},
 'activities': {'main': [{'activityGroup': 'Activités TVA',
    'description': "Distribution d'électricité",
    'naceVersion': '2008'},
   {'activityGroup': 'Activités TVA',
    'description': 'Distribution d’électricité',
    'naceVersion': '2025'},
   {'activityGroup': 'Activités ONSS',
    'description': "Distribution d'électricité",
    'naceVersion': '2008'},
   {'activityGroup': 'Activités ONSS',
    'description': 'Distribution d’électricité',
    'naceVersion': '2025'},
   {'activityGroup': 'Activités TVA',
    'description': "Production d'électricité",
    'naceVersion': '2003'}],
  'secondary': [{'activityGro

## 7. Établissements : mêmes règles, en plus léger

Un établissement a ses propres dénominations/adresses/contacts/activités, à nettoyer avec **exactement les mêmes règles** que ci-dessus. Seule différence avec le document entreprise : pas d'`EnterpriseNumber` (déjà le document de cette entreprise, ce serait une redondance pure). Résultat keyé par `EstablishmentNumber`.

In [12]:
DETAIL_BUILDERS = {
    "denominations": build_denominations,
    "addresses": build_addresses,
    "contacts": build_contacts,
    "activities": build_activities,
}


def build_details(entity_number: str, include=("denominations", "addresses", "contacts", "activities")) -> dict:
    """Construit le sous-ensemble demandé de {denominations, addresses, contacts,
    activities} pour une entité donnée. `include` permet aux établissements de
    prendre les 4, et aux succursales de n'en retenir que 2 (question 8)."""
    return {key: DETAIL_BUILDERS[key](entity_number) for key in include}


def build_establishments(enterprise_number: str) -> dict:
    """{EstablishmentNumber: {startDate?, denominations, addresses, contacts, activities}},
    sans EnterpriseNumber (redondant : c'est déjà le document de cette entreprise)."""
    out = {}
    for establishment in db.establishment.find({"EnterpriseNumber": enterprise_number}):
        entry = {}
        if establishment.get("StartDate"):
            entry["startDate"] = establishment["StartDate"]
        entry.update(build_details(establishment["EstablishmentNumber"]))
        out[establishment["EstablishmentNumber"]] = entry
    return out


pprint(build_establishments("0201.311.226"))


{'2.158.307.210': {'activities': {'main': [{'activityGroup': 'Activités ONSS',
                                            'description': 'Commerce '
                                                           'd’électricité',
                                            'naceVersion': '2025'},
                                           {'activityGroup': 'Activités '
                                                             'ONSSAPL',
                                            'description': 'Distribution et '
                                                           'commerce '
                                                           "d'électricité",
                                            'naceVersion': '2003'},
                                           {'activityGroup': 'Activités ONSS',
                                            'description': 'Commerce '
                                                           "d'électricité",
                                         

In [13]:
# Repère : sortie attendue à ce stade, sur un exemple connu
# (comparez-la visuellement au résultat de la cellule précédente)
{
  "_id": "0201.311.226",
  "enterpriseNumber": "0201.311.226",
  "startDate": "01-01-1968",
  "denominations": {
    "Dénomination": {
      "language": "néerlandais",
      "denomination": "FLUVIUS"
    }
  },
  "addresses": {
    "Siège": {
      "country": "Belgique",
      "zipcode": "3500",
      "municipality": "Hasselt",
      "street": "Trichterheideweg",
      "houseNumber": "8",
      "box": ""
    }
  },
  "contacts": {},
  "activities": {
    "main": [
      {
        "activityGroup": "Activités TVA",
        "description": "Distribution d'électricité",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Distribution d’électricité",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités ONSS",
        "description": "Distribution d'électricité",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités ONSS",
        "description": "Distribution d’électricité",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Production d'électricité",
        "naceVersion": "2003"
      }
    ],
    "secondary": [
      {
        "activityGroup": "Activités TVA",
        "description": "Activités de télécommunications filaires, sans fil et satellitaires",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Production d’électricité à partir de sources non renouvelables",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Collecte et traitement des eaux usées",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Distribution de combustibles gazeux par conduites",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Production d'électricité",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Télécommunications sans fil",
        "naceVersion": "2008"
      }
    ]
  },
  "establishments": {
    "2.158.307.210": {
      "startDate": "01-01-1968",
      "denominations": {
        "Dénomination commerciale": {
          "language": "néerlandais",
          "denomination": "FLUVIUS o.v."
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "3500",
          "municipality": "Hasselt",
          "street": "Trichterheideweg",
          "houseNumber": "8",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Distribution et commerce d'électricité",
            "naceVersion": "2003"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Commerce d’électricité",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Commerce d'électricité",
            "naceVersion": "2008"
          }
        ],
        "secondary": []
      }
    }
  },
  "branches": {},
  "juridicalForm": "Association chargée de mission (Région flamande)",
  "juridicalSituation": "Situation normale",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

{'_id': '0201.311.226',
 'enterpriseNumber': '0201.311.226',
 'startDate': '01-01-1968',
 'denominations': {'Dénomination': {'language': 'néerlandais',
   'denomination': 'FLUVIUS'}},
 'addresses': {'Siège': {'country': 'Belgique',
   'zipcode': '3500',
   'municipality': 'Hasselt',
   'street': 'Trichterheideweg',
   'houseNumber': '8',
   'box': ''}},
 'contacts': {},
 'activities': {'main': [{'activityGroup': 'Activités TVA',
    'description': "Distribution d'électricité",
    'naceVersion': '2008'},
   {'activityGroup': 'Activités TVA',
    'description': 'Distribution d’électricité',
    'naceVersion': '2025'},
   {'activityGroup': 'Activités ONSS',
    'description': "Distribution d'électricité",
    'naceVersion': '2008'},
   {'activityGroup': 'Activités ONSS',
    'description': 'Distribution d’électricité',
    'naceVersion': '2025'},
   {'activityGroup': 'Activités TVA',
    'description': "Production d'électricité",
    'naceVersion': '2003'}],
  'secondary': [{'activityGro

## 8. Succursales : encore plus léger

Une succursale n'a jamais de dénomination propre ni d'activité propre : ne nettoyer que `addresses` et `contacts`. Ni `EnterpriseNumber`, ni `denominations`, ni `activities` dans la sortie. Résultat keyé par `Id`.

In [14]:
def build_branches(enterprise_number: str) -> dict:
    """{Id: {startDate?, addresses, contacts}} : jamais de denominations ni
    d'activities (une succursale n'en a jamais), et jamais d'EnterpriseNumber."""
    out = {}
    for branch in db.branch.find({"EnterpriseNumber": enterprise_number}):
        entry = {}
        if branch.get("StartDate"):
            entry["startDate"] = branch["StartDate"]
        entry.update(build_details(branch["Id"], include=("addresses", "contacts")))
        out[branch["Id"]] = entry
    return out


pprint(build_branches("0257.883.408"))


{'9.000.006.626': {'addresses': {'Succursale': {'country': 'Belgique',
                                                'houseNumber': '28',
                                                'municipality': 'Bruxelles',
                                                'street': 'Rue de la Loi',
                                                'zipcode': '1040'}},
                   'contacts': {},
                   'startDate': '01-09-1995'}}


In [15]:
# Repère : sortie attendue à ce stade, sur un exemple connu
# (comparez-la visuellement au résultat de la cellule précédente)
{
  "_id": "0257.883.408",
  "enterpriseNumber": "0257.883.408",
  "startDate": "01-09-1995",
  "denominations": {
    "Dénomination": {
      "language": "français",
      "denomination": "ASSOCIATION TURQUE DES EXPORTATEURS DE TEXTILE ET D'HABILLEMENT D'ISTANBUL - ITKIB"
    }
  },
  "addresses": {
    "Siège": {
      "country": "Turquie",
      "zipcode": "34196",
      "municipality": "yenibosna - Istamboul",
      "street": "itkib bis ticaret komplexi b/blok coban cesme mekvil sanayi/caddesi",
      "houseNumber": "0",
      "box": ""
    }
  },
  "contacts": {},
  "activities": {
    "main": [],
    "secondary": []
  },
  "establishments": {
    "2.076.372.003": {
      "startDate": "02-05-1996",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "ITKIB"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "1040",
          "municipality": "Bruxelles",
          "street": "Rue de la Loi",
          "houseNumber": "28",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités",
            "description": "Agences de presse",
            "naceVersion": "2003"
          },
          {
            "activityGroup": "Activités",
            "description": "Activités d’agence de presse",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    }
  },
  "branches": {
    "9.000.006.626": {
      "startDate": "01-09-1995",
      "addresses": {
        "Succursale": {
          "country": "Belgique",
          "zipcode": "1040",
          "municipality": "Bruxelles",
          "street": "Rue de la Loi",
          "houseNumber": "28",
          "box": ""
        }
      },
      "contacts": {}
    }
  },
  "juridicalForm": "Entité étrangère",
  "juridicalSituation": "Situation normale",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

{'_id': '0257.883.408',
 'enterpriseNumber': '0257.883.408',
 'startDate': '01-09-1995',
 'denominations': {'Dénomination': {'language': 'français',
   'denomination': "ASSOCIATION TURQUE DES EXPORTATEURS DE TEXTILE ET D'HABILLEMENT D'ISTANBUL - ITKIB"}},
 'addresses': {'Siège': {'country': 'Turquie',
   'zipcode': '34196',
   'municipality': 'yenibosna - Istamboul',
   'street': 'itkib bis ticaret komplexi b/blok coban cesme mekvil sanayi/caddesi',
   'houseNumber': '0',
   'box': ''}},
 'contacts': {},
 'activities': {'main': [], 'secondary': []},
 'establishments': {'2.076.372.003': {'startDate': '02-05-1996',
   'denominations': {'Dénomination commerciale': {'language': 'français',
     'denomination': 'ITKIB'}},
   'addresses': {"Unité d'établissement": {'country': 'Belgique',
     'zipcode': '1040',
     'municipality': 'Bruxelles',
     'street': 'Rue de la Loi',
     'houseNumber': '28',
     'box': ''}},
   'contacts': {},
   'activities': {'main': [{'activityGroup': 'Activité

## 9. Écriture dans `entreprise_silver`

Snapshot complet : vider `entreprise_silver` puis écrire le résultat.

In [16]:
def build_silver_document(enterprise_doc: dict) -> dict:
    """Assemble le document silver complet d'une entreprise : en-tête (question 2),
    ses propres denominations/addresses/contacts/activities (questions 3 à 6),
    puis ses établissements (question 7) et ses succursales (question 8)."""
    enterprise_number = enterprise_doc["_id"]
    silver_doc = build_silver_header(enterprise_doc)
    silver_doc.update(build_details(enterprise_number))
    silver_doc["establishments"] = build_establishments(enterprise_number)
    silver_doc["branches"] = build_branches(enterprise_number)
    return silver_doc


def refresh_entreprise_silver(batch_size: int = 500):
    """Snapshot complet : vide `entreprise_silver`, puis réécrit un document par
    entreprise, par lots de `batch_size` (chaque document coûte plusieurs requêtes,
    d'où un lot plus petit que dans la couche bronze)."""
    db.entreprise_silver.delete_many({})
    batch = []
    for enterprise_doc in db.enterprise.find({}):
        batch.append(build_silver_document(enterprise_doc))
        if len(batch) >= batch_size:
            db.entreprise_silver.insert_many(batch)
            batch = []
    if batch:
        db.entreprise_silver.insert_many(batch)


refresh_entreprise_silver()
print(f"entreprise_silver: {db.entreprise_silver.count_documents({}):,} documents")


entreprise_silver: 1,955,776 documents


## 10. Vérification

Comparer un document silver produit à ce que prédit la spec ci-dessus, sur une entreprise connue.

In [17]:
produced = db.entreprise_silver.find_one({"_id": "0201.105.843"})
pprint(produced)


{'_id': '0201.105.843',
 'activities': {'main': [{'activityGroup': 'Activités ONSS',
                          'description': 'Administration de et contribution à '
                                         'l’amélioration de l’efficacité des '
                                         'activités économiques',
                          'naceVersion': '2025'},
                         {'activityGroup': 'Activités TVA',
                          'description': 'Études de marché et sondages',
                          'naceVersion': '2025'},
                         {'activityGroup': 'Activités TVA',
                          'description': "Bureau d'étude de marché",
                          'naceVersion': '2003'},
                         {'activityGroup': 'Activités ONSS',
                          'description': 'Administration publique (tutelle) '
                                         'des activités économiques',
                          'naceVersion': '2008'},
                   

In [ ]:
# Repère : sortie attendue à ce stade, sur un exemple connu
# (comparez-la visuellement au résultat de la cellule précédente)
{
  "_id": "0201.105.843",
  "enterpriseNumber": "0201.105.843",
  "startDate": "02-03-1956",
  "denominations": {
    "Abréviation": {
      "language": "français",
      "denomination": "I.D.E.A."
    },
    "Dénomination": {
      "language": "français",
      "denomination": "\"I.D.E.A. S.C\""
    }
  },
  "addresses": {
    "Siège": {
      "country": "Belgique",
      "zipcode": "7000",
      "municipality": "Mons",
      "street": "Rue de Nimy",
      "houseNumber": "53",
      "box": ""
    }
  },
  "contacts": {
    "email": "officiel.ic-idea@idea.be"
  },
  "activities": {
    "main": [
      {
        "activityGroup": "Activités ONSS",
        "description": "Administration de et contribution à l’amélioration de l’efficacité des activités économiques",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités ONSS",
        "description": "Administration publique (tutelle) des activités économiques",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Études de marché et sondages",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Études de marché et sondages d'opinion",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Bureau d'étude de marché",
        "naceVersion": "2003"
      }
    ],
    "secondary": [
      {
        "activityGroup": "Activités TVA",
        "description": "Travaux de dragage",
        "naceVersion": "2025"
      }
    ]
  },
  "establishments": {
    "2.382.100.462": {
      "startDate": "01-01-2026",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Distribution d'eau"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7011",
          "municipality": "Mons",
          "street": "Rue de Baudour (G.)",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Captage, traitement et distribution d’eau",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.300.665.301": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d'épuration de Braine-Le-Comte"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7090",
          "municipality": "Braine-le-Comte",
          "street": "Route de Petit Roeulx",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {
        "email": "ludovic.delhaye@idea.be",
        "phone": "065/37 57 27"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.343.255.427": {
      "startDate": "01-01-2023",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d’épuration Ecaussinnes"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7190",
          "municipality": "Ecaussinnes",
          "street": "Rue de l'Avedelle",
          "houseNumber": "sn",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.162.854.629": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "La Maison de l'Entreprise - Mons"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7000",
          "municipality": "Mons",
          "street": "Rue René Descartes",
          "houseNumber": "2",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Autres activités de service de soutien aux entreprises nca",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Intercommunales à vocation générale",
            "naceVersion": "2003"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Autres activités de soutien aux entreprises n.c.a.",
            "naceVersion": "2008"
          }
        ],
        "secondary": []
      }
    },
    "2.300.510.990": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d'épuration de Chapelle-Lez-Herlaimont"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7160",
          "municipality": "Chapelle-lez-Herlaimont",
          "street": "Rue du Vent de Bise",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {
        "email": "ludovic.delhaye@idea.be",
        "phone": "065/37 57 27"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.382.106.697": {
      "startDate": "01-01-2026",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "La Maison de l'Entreprise"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7100",
          "municipality": "La Louvière",
          "street": "Rue Arthur Delaby(L.L)",
          "houseNumber": "5",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Autres activités de service de soutien aux entreprises nca",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.165.787.195": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Garocentre"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7110",
          "municipality": "La Louvière",
          "street": "Rue Athéna(H-G)",
          "houseNumber": "-",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration de et contribution à l’amélioration de l’efficacité des activités économiques",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Intercommunales à vocation générale",
            "naceVersion": "2003"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration publique (tutelle) des activités économiques",
            "naceVersion": "2008"
          }
        ],
        "secondary": []
      }
    },
    "2.162.853.936": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "I.D.E.A. Mons-Borinage-Centre"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7100",
          "municipality": "La Louvière",
          "street": "Rue Hamoir(L.L)",
          "houseNumber": "30",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Autres activités de télécommunication, y compris télédistribution",
            "naceVersion": "2003"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Activités de télécommunications filaires, sans fil et satellitaires",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Télécommunications sans fil",
            "naceVersion": "2008"
          }
        ],
        "secondary": []
      }
    },
    "2.300.628.083": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d'épuaration de Boussoit"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7110",
          "municipality": "La Louvière",
          "street": "Rue de Thieu(BO)",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {
        "phone": "065/37 57 27",
        "email": "ludovic.delhaye@idea.be"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.300.626.994": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d'épuration de Quièvrain"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7380",
          "municipality": "Quiévrain",
          "street": "Rue du Bruil",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {
        "email": "ludovic.delhaye@idea.be",
        "phone": "065/37 57 27"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.300.627.489": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d'épuration de Saint-Vaast"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7100",
          "municipality": "La Louvière",
          "street": "Rue du Moulin à Eau(S-V)",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {
        "email": "ludovic.delhaye@idea.be",
        "phone": "065/37 57 27"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.300.492.679": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "IDEA - UMH"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7000",
          "municipality": "Mons",
          "street": "Boulevard Initialis",
          "houseNumber": "30",
          "box": ""
        }
      },
      "contacts": {
        "email": "ludovic.delhaye@idea.be",
        "phone": "065/37 57 27"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration de et contribution à l’amélioration de l’efficacité des activités économiques",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration publique (tutelle) des activités économiques",
            "naceVersion": "2008"
          }
        ],
        "secondary": []
      }
    },
    "2.162.853.540": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "I.D.E.A. Région Mons-Borinage-Centre"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7180",
          "municipality": "Seneffe",
          "street": "Rue de Soudromont",
          "houseNumber": "1",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2003"
          }
        ],
        "secondary": []
      }
    },
    "2.300.627.885": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d'épuration de Trivières"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7100",
          "municipality": "La Louvière",
          "street": "Rue du Provia(TRI)",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {
        "email": "ludovic.delhaye@idea.be",
        "phone": "065/37 57 27"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.343.255.130": {
      "startDate": "01-01-2023",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d’épuration Dour"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7370",
          "municipality": "Dour",
          "street": "Rue de Baisieux",
          "houseNumber": "sn",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.343.255.823": {
      "startDate": "01-01-2023",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d’épuration Morlanwelz"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7140",
          "municipality": "Morlanwelz",
          "street": "Rue des Haignies(MLZ)",
          "houseNumber": "sn",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.162.853.342": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "I.D.E.A. Région Mons-Borinage-Centre"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7390",
          "municipality": "Quaregnon",
          "street": "Chasse des Prés",
          "houseNumber": "1",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2003"
          }
        ],
        "secondary": []
      }
    },
    "2.162.852.253": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "I.D.E.A. Région Mons-Borinage-Centre"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7000",
          "municipality": "Mons",
          "street": "Rue de Nimy",
          "houseNumber": "53",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration publique (tutelle) des activités économiques",
            "naceVersion": "2008"
          },
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Intercommunales à vocation générale",
            "naceVersion": "2003"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration de et contribution à l’amélioration de l’efficacité des activités économiques",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.343.255.724": {
      "startDate": "01-01-2023",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d’épuration Godarville"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7160",
          "municipality": "Chapelle-lez-Herlaimont",
          "street": "Rue du Castia",
          "houseNumber": "sn",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.162.854.926": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "La Maison de l'Entreprise - Binche"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7130",
          "municipality": "Binche",
          "street": "Rue des Pastures(BIN)",
          "houseNumber": "95",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Autres activités de service de soutien aux entreprises nca",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Autres activités de soutien aux entreprises n.c.a.",
            "naceVersion": "2008"
          },
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Intercommunales à vocation générale",
            "naceVersion": "2003"
          }
        ],
        "secondary": []
      }
    },
    "2.343.255.328": {
      "startDate": "01-01-2023",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d’épuration Frameries"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7080",
          "municipality": "Frameries",
          "street": "Rue des Fours-à-Chaux",
          "houseNumber": "sn",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.300.627.291": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d'épuration de Soignies Biamont"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7060",
          "municipality": "Soignies",
          "street": "Chemin de la Platinerie",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {
        "email": "ludovic.delhaye@idea.be",
        "phone": "065/37 57 27"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.162.853.639": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "I.D.E.A. Région Mons-Borinage-Centre"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7110",
          "municipality": "La Louvière",
          "street": "Rue Grand Peuplier(H-A)",
          "houseNumber": "20",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Télécommunications sans fil",
            "naceVersion": "2008"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Activités de télécommunications filaires, sans fil et satellitaires",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Autres activités de télécommunication, y compris télédistribution",
            "naceVersion": "2003"
          }
        ],
        "secondary": []
      }
    },
    "2.227.021.515": {
      "startDate": "01-01-2014",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Service entretien des biens - Infrastructures"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7033",
          "municipality": "Mons",
          "street": "Rue de Ciply (C.)",
          "houseNumber": "265",
          "box": "B"
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration de et contribution à l’amélioration de l’efficacité des activités économiques",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration publique (tutelle) des activités économiques",
            "naceVersion": "2008"
          }
        ],
        "secondary": []
      }
    }
  },
  "branches": {},
  "juridicalForm": "Société coopérative",
  "juridicalSituation": "Situation normale",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

## 12. Schéma cible de `entreprise_silver`

| Champ | Type | Origine / règle |
|---|---|---|
| `_id` | string | copié tel quel du bronze |
| `enterpriseNumber` | string | `EnterpriseNumber` (ou `_id` en secours) |
| `startDate` | string, optionnel | copié si présent |
| `status`, `juridicalSituation`, `typeOfEnterprise`, `juridicalForm`, `juridicalFormCAC` | string, optionnels | traduits FR via `code.csv`, omis si absents du bronze |
| `denominations` | dict `{type traduit: {language, denomination}}` | dernier gagne en cas de type dupliqué |
| `addresses` | dict `{type traduit: {country, zipcode, municipality, street, houseNumber, box}}` | champs vides omis, `country` nettoyé + défaut `"Belgique"` |
| `contacts` | dict `{email?, phone?, web?}` | `EntityContact` jamais lu |
| `activities` | `{main: [...], secondary: [...]}` | dédoublonné par `(activityGroup, description)`, version NACE la plus récente gagne, `naceCode` brut jamais gardé |
| `establishments` | dict `{EstablishmentNumber: {startDate?, denominations, addresses, contacts, activities}}` | mêmes règles que l'entreprise, sans `EnterpriseNumber` |
| `branches` | dict `{Id: {startDate?, addresses, contacts}}` | sans `denominations` ni `activities` (une succursale n'en a jamais) |